
# Roxy notebook example: Residue separation and spacing descriptors

This notebook is a **reference implementation example** for the **residue separation and spacing descriptor family** in Roxy.

These descriptors quantify **how far apart residues or residue groups are distributed along the sequence**. They are useful because two sequences can have very similar composition, but very different spatial organization in sequence space.

## Covered outputs

This notebook implements examples such as:

- mean spacing between residues of interest
- median spacing between residues of interest
- minimum and maximum spacing
- normalized spacing summaries
- spacing variance
- spacing between residue groups
- spacing between aromatic / charged / hydrophobic residues
- nearest-neighbor spacing summaries
- terminal-to-terminal span of residue groups
- class-style implementation for later migration into Roxy

The goal is to provide a **clean teaching implementation** that can later become a real `spacing.py` or related module in Roxy.


In [1]:

import numpy as np
import pandas as pd


## Demo dataset

In [2]:

df_demo = pd.DataFrame(
    {
        "sequence_id": [
            "space_1",
            "space_2",
            "space_3",
            "space_4",
            "space_5",
            "space_6",
        ],
        "sequence": [
            "MKWVTFISLLFLFSSAYSRGVFRR",
            "GGGGGGGGGGGGGGG",
            "KRRKRRKRRKRRDDDDEE",
            "ACDEFGHIKLMNPQRSTVWY",
            "PPPPGSSSSSTTTTNNQQQ",
            "MSTNPKPQRITLKDGNKVELV",
        ],
        "label": ["A", "B", "A", "B", "A", "B"],
    }
)

df_demo


,sequence_id,sequence,label
0,space_1,MKWVTFISLLFLFSSAYSRGVFRR,A
1,space_2,GGGGGGGGGGGGGGG,B
2,space_3,KRRKRRKRRKRRDDDDEE,A
3,space_4,ACDEFGHIKLMNPQRSTVWY,B
4,space_5,PPPPGSSSSSTTTTNNQQQ,A
5,space_6,MSTNPKPQRITLKDGNKVELV,B


## Constants

In [3]:

STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")

AA_GROUPS = {
    "positive": set("KRH"),
    "negative": set("DE"),
    "charged": set("KRHDE"),
    "polar": set("STNQCYWHKRDE"),
    "nonpolar": set("AVLIMFGP"),
    "aromatic": set("FWYH"),
    "aliphatic": set("AVLIM"),
    "hydrophobic": set("AVLIMFWCY"),
    "hydrophilic": set("RNDQEHKST"),
    "disorder_promoting": set("ARGQSEPK"),
    "order_promoting": set("CWYFILNV"),
}

AA_SINGLETS = {
    "K": set("K"),
    "R": set("R"),
    "D": set("D"),
    "E": set("E"),
    "G": set("G"),
    "P": set("P"),
    "W": set("W"),
    "Y": set("Y"),
}


## Helper functions

In [4]:

def clean_sequence(seq: str) -> str:
    if pd.isna(seq):
        return ""
    seq = str(seq).strip().upper().replace("*", "")
    return "".join([aa for aa in seq if aa in STANDARD_AA])


def positions_of_group(seq: str, aa_group):
    return [i for i, aa in enumerate(seq) if aa in aa_group]


def inter_event_distances(positions):
    if len(positions) < 2:
        return []
    return list(np.diff(positions))


def nearest_neighbor_distances(positions):
    if len(positions) < 2:
        return []
    dists = []
    for i, pos in enumerate(positions):
        neighbors = []
        if i > 0:
            neighbors.append(pos - positions[i - 1])
        if i < len(positions) - 1:
            neighbors.append(positions[i + 1] - pos)
        dists.append(min(neighbors))
    return dists


def safe_stats(values):
    if len(values) == 0:
        return {
            "mean": np.nan,
            "median": np.nan,
            "min": np.nan,
            "max": np.nan,
            "std": np.nan,
        }
    return {
        "mean": float(np.mean(values)),
        "median": float(np.median(values)),
        "min": float(np.min(values)),
        "max": float(np.max(values)),
        "std": float(np.std(values, ddof=0)),
    }


def normalized_stat(value, seq_len):
    if seq_len == 0 or np.isnan(value):
        return np.nan
    return value / seq_len


def span_of_positions(positions):
    if len(positions) < 2:
        return np.nan
    return float(positions[-1] - positions[0])


def mean_cross_group_distance(seq: str, group_a, group_b):
    pos_a = positions_of_group(seq, group_a)
    pos_b = positions_of_group(seq, group_b)
    if len(pos_a) == 0 or len(pos_b) == 0:
        return np.nan

    distances = []
    for a in pos_a:
        distances.append(min(abs(a - b) for b in pos_b))
    return float(np.mean(distances))


def density_of_positions(positions, seq_len):
    if seq_len == 0:
        return np.nan
    return len(positions) / seq_len


## Core descriptor function

In [5]:

def spacing_descriptors(seq: str) -> dict:
    seq = clean_sequence(seq)
    seq_len = len(seq)

    out = {
        "space_length": seq_len,
        "space_valid_residue_count": seq_len,
    }

    if seq_len == 0:
        return out

    tracked_groups = {
        "charged": AA_GROUPS["charged"],
        "hydrophobic": AA_GROUPS["hydrophobic"],
        "aromatic": AA_GROUPS["aromatic"],
        "polar": AA_GROUPS["polar"],
        "positive": AA_GROUPS["positive"],
        "negative": AA_GROUPS["negative"],
        "gly": AA_SINGLETS["G"],
        "pro": AA_SINGLETS["P"],
    }

    for name, group in tracked_groups.items():
        positions = positions_of_group(seq, group)
        dists = inter_event_distances(positions)
        nn_dists = nearest_neighbor_distances(positions)
        dist_stats = safe_stats(dists)
        nn_stats = safe_stats(nn_dists)
        span = span_of_positions(positions)

        out[f"space_{name}_count"] = len(positions)
        out[f"space_{name}_density"] = density_of_positions(positions, seq_len)
        out[f"space_{name}_span"] = span
        out[f"space_{name}_span_norm"] = normalized_stat(span, seq_len)

        out[f"space_{name}_mean"] = dist_stats["mean"]
        out[f"space_{name}_median"] = dist_stats["median"]
        out[f"space_{name}_min"] = dist_stats["min"]
        out[f"space_{name}_max"] = dist_stats["max"]
        out[f"space_{name}_std"] = dist_stats["std"]

        out[f"space_{name}_mean_norm"] = normalized_stat(dist_stats["mean"], seq_len)
        out[f"space_{name}_median_norm"] = normalized_stat(dist_stats["median"], seq_len)
        out[f"space_{name}_max_norm"] = normalized_stat(dist_stats["max"], seq_len)

        out[f"space_{name}_nn_mean"] = nn_stats["mean"]
        out[f"space_{name}_nn_median"] = nn_stats["median"]
        out[f"space_{name}_nn_min"] = nn_stats["min"]
        out[f"space_{name}_nn_max"] = nn_stats["max"]

    # Cross-group spacing summaries
    out["space_positive_negative_cross_mean"] = mean_cross_group_distance(
        seq, AA_GROUPS["positive"], AA_GROUPS["negative"]
    )
    out["space_charged_hydrophobic_cross_mean"] = mean_cross_group_distance(
        seq, AA_GROUPS["charged"], AA_GROUPS["hydrophobic"]
    )
    out["space_aromatic_polar_cross_mean"] = mean_cross_group_distance(
        seq, AA_GROUPS["aromatic"], AA_GROUPS["polar"]
    )
    out["space_disorder_order_cross_mean"] = mean_cross_group_distance(
        seq, AA_GROUPS["disorder_promoting"], AA_GROUPS["order_promoting"]
    )

    return out


## Functional usage on one sequence

In [6]:

example = spacing_descriptors(df_demo.loc[0, "sequence"])
list(example.items())[:18]


[('space_length', 24),
 ('space_valid_residue_count', 24),
 ('space_charged_count', 4),
 ('space_charged_density', 0.16666666666666666),
 ('space_charged_span', 22.0),
 ('space_charged_span_norm', 0.9166666666666666),
 ('space_charged_mean', 7.333333333333333),
 ('space_charged_median', 4.0),
 ('space_charged_min', 1.0),
 ('space_charged_max', 17.0),
 ('space_charged_std', 6.944222218666553),
 ('space_charged_mean_norm', 0.3055555555555555),
 ('space_charged_median_norm', 0.16666666666666666),
 ('space_charged_max_norm', 0.7083333333333334),
 ('space_charged_nn_mean', 5.75),
 ('space_charged_nn_median', 2.5),
 ('space_charged_nn_min', 1.0),
 ('space_charged_nn_max', 17.0)]

## Apply spacing descriptors to the full dataset

In [7]:

df_space = pd.concat(
    [
        df_demo,
        df_demo["sequence"].apply(spacing_descriptors).apply(pd.Series),
    ],
    axis=1,
)

df_space.head()


,sequence_id,sequence,label,space_length,space_valid_residue_count,space_charged_count,space_charged_density,space_charged_span,space_charged_span_norm,space_charged_mean,...,space_pro_median_norm,space_pro_max_norm,space_pro_nn_mean,space_pro_nn_median,space_pro_nn_min,space_pro_nn_max,space_positive_negative_cross_mean,space_charged_hydrophobic_cross_mean,space_aromatic_polar_cross_mean,space_disorder_order_cross_mean
0,space_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24.0,24.0,4.0,0.166667,22.0,0.916667,7.333333,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.5,1.00,1.300000
1,space_2,GGGGGGGGGGGGGGG,B,15.0,15.0,0.0,0.000000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,space_3,KRRKRRKRRKRRDDDDEE,A,18.0,18.0,18.0,1.000000,17.0,0.944444,1.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,6.500000,NaN,NaN,NaN
3,space_4,ACDEFGHIKLMNPQRSTVWY,B,20.0,20.0,5.0,0.250000,12.0,0.600000,3.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,6.333333,1.4,0.25,1.500000
4,space_5,PPPPGSSSSSTTTTNNQQQ,A,19.0,19.0,0.0,0.000000,NaN,NaN,NaN,...,0.052632,0.052632,1.0,1.0,1.0,1.0,NaN,NaN,NaN,7.769231


## Inspect spacing descriptor columns

In [8]:

space_cols = [c for c in df_space.columns if c.startswith("space_") and c not in {"space_length", "space_valid_residue_count"}]
len(space_cols), space_cols[:16]


(132,
 ['space_charged_count',
  'space_charged_density',
  'space_charged_span',
  'space_charged_span_norm',
  'space_charged_mean',
  'space_charged_median',
  'space_charged_min',
  'space_charged_max',
  'space_charged_std',
  'space_charged_mean_norm',
  'space_charged_median_norm',
  'space_charged_max_norm',
  'space_charged_nn_mean',
  'space_charged_nn_median',
  'space_charged_nn_min',
  'space_charged_nn_max'])

In [9]:

df_space[
    [
        "sequence_id",
        "space_charged_mean",
        "space_charged_median",
        "space_hydrophobic_mean",
        "space_aromatic_span",
        "space_positive_negative_cross_mean",
        "space_gly_density",
        "space_pro_density",
    ]
]


,sequence_id,space_charged_mean,space_charged_median,space_hydrophobic_mean,space_aromatic_span,space_positive_negative_cross_mean,space_gly_density,space_pro_density
0,space_1,7.333333,4.0,1.615385,19.0,NaN,0.041667,0.000000
1,space_2,NaN,NaN,NaN,NaN,NaN,1.000000,0.000000
2,space_3,1.000000,1.0,NaN,NaN,6.500000,0.000000,0.000000
3,space_4,3.000000,2.5,2.375000,15.0,6.333333,0.050000,0.050000
4,space_5,NaN,NaN,NaN,NaN,NaN,0.052632,0.210526
5,space_6,2.600000,3.0,4.000000,NaN,4.000000,0.047619,0.095238


## Dataset-level summary

In [10]:

space_summary = (
    df_space[space_cols]
    .mean(axis=0, numeric_only=True)
    .sort_values(ascending=False)
    .rename("mean_value")
    .reset_index()
    .rename(columns={"index": "descriptor"})
)

space_summary.head(15)


,descriptor,mean_value
0,space_hydrophobic_span,20.000000
1,space_polar_span,17.400000
2,space_aromatic_span,17.000000
3,space_charged_span,16.000000
4,space_gly_span,14.000000
5,space_positive_span,13.000000
6,space_polar_count,11.166667
7,space_aromatic_max,8.500000
8,space_positive_max,7.000000
9,space_positive_nn_max,7.000000


## Sanity checks

In [11]:

assert "space_charged_mean" in df_space.columns
assert "space_hydrophobic_median" in df_space.columns
assert "space_aromatic_span" in df_space.columns
assert "space_positive_negative_cross_mean" in df_space.columns
assert "space_gly_density" in df_space.columns
assert df_space["space_length"].min() > 0

print(f"Number of spacing descriptor columns: {len(space_cols)}")
print("Spacing descriptor checks passed.")


Number of spacing descriptor columns: 132
Spacing descriptor checks passed.


## Class-style implementation closer to the real package

In [12]:

class ResidueSpacingDescriptors:
    """Example class-style spacing implementation for later migration into Roxy."""

    def transform_sequence(self, seq: str) -> dict:
        return spacing_descriptors(seq)

    def transform(self, sequences) -> pd.DataFrame:
        return pd.DataFrame([self.transform_sequence(seq) for seq in sequences])


space_transformer = ResidueSpacingDescriptors()
space_matrix = space_transformer.transform(df_demo["sequence"].tolist())
space_matrix.head()


,space_length,space_valid_residue_count,space_charged_count,space_charged_density,space_charged_span,space_charged_span_norm,space_charged_mean,space_charged_median,space_charged_min,space_charged_max,...,space_pro_median_norm,space_pro_max_norm,space_pro_nn_mean,space_pro_nn_median,space_pro_nn_min,space_pro_nn_max,space_positive_negative_cross_mean,space_charged_hydrophobic_cross_mean,space_aromatic_polar_cross_mean,space_disorder_order_cross_mean
0,24,24,4,0.166667,22.0,0.916667,7.333333,4.0,1.0,17.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.5,1.00,1.300000
1,15,15,0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,18,18,18,1.000000,17.0,0.944444,1.000000,1.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,6.500000,NaN,NaN,NaN
3,20,20,5,0.250000,12.0,0.600000,3.000000,2.5,1.0,6.0,...,NaN,NaN,NaN,NaN,NaN,NaN,6.333333,1.4,0.25,1.500000
4,19,19,0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,...,0.052632,0.052632,1.0,1.0,1.0,1.0,NaN,NaN,NaN,7.769231


## Merge transformer output back to the dataset

In [13]:

df_space_class = pd.concat([df_demo, space_matrix], axis=1)
df_space_class.head()


,sequence_id,sequence,label,space_length,space_valid_residue_count,space_charged_count,space_charged_density,space_charged_span,space_charged_span_norm,space_charged_mean,...,space_pro_median_norm,space_pro_max_norm,space_pro_nn_mean,space_pro_nn_median,space_pro_nn_min,space_pro_nn_max,space_positive_negative_cross_mean,space_charged_hydrophobic_cross_mean,space_aromatic_polar_cross_mean,space_disorder_order_cross_mean
0,space_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24,24,4,0.166667,22.0,0.916667,7.333333,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.5,1.00,1.300000
1,space_2,GGGGGGGGGGGGGGG,B,15,15,0,0.000000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,space_3,KRRKRRKRRKRRDDDDEE,A,18,18,18,1.000000,17.0,0.944444,1.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,6.500000,NaN,NaN,NaN
3,space_4,ACDEFGHIKLMNPQRSTVWY,B,20,20,5,0.250000,12.0,0.600000,3.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,6.333333,1.4,0.25,1.500000
4,space_5,PPPPGSSSSSTTTTNNQQQ,A,19,19,0,0.000000,NaN,NaN,NaN,...,0.052632,0.052632,1.0,1.0,1.0,1.0,NaN,NaN,NaN,7.769231



## Suggested next refactor into the package

A clean migration path into Roxy would be:

- move helper logic into `roxy/sequence/order.py` or a dedicated `spacing.py`
- keep residue groups in `roxy/core/constants.py`
- expose a class such as `ResidueSpacingDescriptors`
- allow configurable:
  - tracked groups
  - residue-specific spacing summaries
  - cross-group spacing summaries
- add tests for:
  - empty sequences
  - sequences with only one residue of a given type
  - clustered vs dispersed arrangements
  - lower-case input
  - invalid characters removed during cleaning


## Optional export

In [ ]:
# df_space.to_csv("demo_spacing_descriptors.csv", index=False)
